# RappiPlus: de los datos a las decisiones de negocio

**Diagnóstico integral de un servicio de delivery: calidad de datos, rentabilidad, funnel de conversión, retención por cohortes, un experimento A/B y un dashboard en Power BI.**

<a href="https://colab.research.google.com/github/fvt1999pipe-eng/analisis_rappiplus/blob/main/analisis_rappiplus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

**Autor:** Felipe Vásquez Torres · [LinkedIn](https://www.linkedin.com/in/felipe-vasquez-torres) · [Portafolio](https://fvt1999pipe-eng.github.io)

## Contexto y objetivo

El objetivo del proyecto es evaluar el desempeño del servicio **RappiPlus** en el primer semestre de 2025 y proponer **recomendaciones accionables** basadas en datos. El análisis responde cinco preguntas en orden: si los datos son confiables, si el negocio es rentable, dónde se pierden los usuarios, si los usuarios regresan y si un cambio en el checkout mejora la conversión.

## Datos

| Fuente | Contenido |
|---|---|
| `rappiplus_orders_raw.csv` | 25,100 pedidos con precios, descuentos y monto total |
| `rappiplus_catalog.csv` | 7 productos con su costo unitario, categoría y proveedor |
| `rappiplus_marketing_spend.csv` | 1,620 registros de inversión en marketing por canal, campaña y país |
| Tablas SQL `events`, `users` y `user_activity` | Comportamiento de los usuarios en la plataforma (PostgreSQL) |
| `experiment_checkout_ui.csv` | 10,000 usuarios de un experimento A/B sobre la interfaz del checkout |

## Contenido

1. Calidad de los datos (Python)
2. Rentabilidad del negocio
3. Funnel de conversión (SQL)
4. Retención por cohortes (SQL)
5. Experimento A/B del checkout
6. Dashboard en Power BI

---
## 1. Calidad de los datos

### 1.1 Carga de datos y vista rápida

In [1]:
# Librerías
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Cargar los datasets
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv")
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv")
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")

In [3]:
# Primeras filas y estructura de cada dataset
orders.head(5)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [4]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [5]:
catalog.head(5)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [6]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [7]:
marketing.head(5)

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [8]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


**Observaciones de la exploración inicial**

**orders (25 100 registros, 12 columnas)**
- `fecha_hora_pedido` está como tipo `object`; debe convertirse a `datetime`.
- Hay valores nulos en varias columnas: `pais` (300), `categoria_producto` (80), `cantidad`, `precio_unitario` y `monto_descuento` (50 c/u), `fuente_referencia` y `nombre_producto` (30 c/u) y `dispositivo` (20). `monto_total` no tiene nulos.
- `cantidad` aparece como `float64`, aunque debería ser un valor entero.
- Las categorías no tienen un formato uniforme frente al catálogo (por ejemplo, "Electronica" en orders y "Electrónica" en catalog), lo que puede afectar la unión de tablas.
- Se debe verificar la existencia de pedidos duplicados y de valores negativos o atípicos en las columnas numéricas.

**catalog (7 registros, 4 columnas)**
- No presenta valores nulos y los tipos de datos son correctos.
- Se debe validar que los nombres de producto coincidan con los registrados en orders.

**marketing (1 620 registros, 5 columnas)**
- `fecha` está como tipo `object`; debe convertirse a `datetime`.
- `canal` tiene 101 valores nulos, que podrían recuperarse a partir de `id_campaña`, ya que este incluye el canal en su nombre.

**Siguiente paso:** analizar por separado las variables numéricas y categóricas con `describe()` para identificar inconsistencias, valores atípicos y el tratamiento adecuado de los nulos.

### 1.2 Revisión y corrección de la calidad de los datos

Se revisan los tres datasets: fechas, variables numéricas (sin negativos ni ceros inválidos), consistencia de los montos, duplicados y variables categóricas.

In [9]:
# Copias de trabajo para conservar los datos originales
orders_clean = orders.copy()
catalog_clean = catalog.copy()
marketing_clean = marketing.copy()

In [10]:
# Revisión y eliminación de duplicados en los tres datasets
print("limpieza duplicados orders_clean")
print(orders_clean.duplicated(subset="id_pedido").sum())
print("Filas antes:", len(orders_clean))
orders_clean = orders_clean.drop_duplicates(subset="id_pedido").reset_index(drop=True)
print("Filas después:", len(orders_clean))
print()


print("limpieza duplicados catalog_clean")
print(catalog_clean.duplicated(subset="nombre_producto").sum())

print()
print("limpieza duplicados marketing_clean")
print(marketing_clean.duplicated(subset=["id_campaña","fecha"]).sum())

limpieza duplicados orders_clean
100
Filas antes: 25100
Filas despues: 25000

limpieza duplicados catalog_clean
0

limpieza duplicados marketing_clean
0


In [11]:
# Conversión de las fechas a datetime
orders_clean["fecha_hora_pedido"] = pd.to_datetime(orders_clean["fecha_hora_pedido"], errors="coerce")
marketing_clean["fecha"] = pd.to_datetime(marketing_clean["fecha"], errors="coerce")


print(orders_clean["fecha_hora_pedido"].dtype)
print(marketing_clean["fecha"].dtype)
print()
print("revision de nulos despues del cambio")
print(orders_clean["fecha_hora_pedido"].isna().sum())
print(marketing_clean["fecha"].isna().sum())
print()
print("rango de fechas de orders_clean")
print(orders_clean["fecha_hora_pedido"].min())
print(orders_clean["fecha_hora_pedido"].max())
print()
print("rango de fechas de marketing_clean")
print(marketing_clean["fecha"].min())
print(marketing_clean["fecha"].max())

datetime64[ns]
datetime64[ns]

revision de nulos despues del cambio
0
0

rango de fechas de orders_clean
2025-01-01 00:00:00
2025-06-30 00:00:00

rango de fechas de marketing_clean
2025-01-01 00:00:00
2025-06-29 00:00:00


In [12]:
# Separación de columnas numéricas y categóricas para revisar su distribución
# orders
orders_num = ["cantidad", "precio_unitario", "monto_descuento", "monto_total"]
orders_cat = ["id_pedido", "id_usuario", "pais", "dispositivo", "fuente_referencia", "nombre_producto", "categoria_producto"]

# catalog
catalog_num = ["costo_unitario"]
catalog_cat = ["nombre_producto", "categoria_producto", "proveedor"]

# marketing
marketing_num = ["gasto"]
marketing_cat = ["pais", "id_campaña", "canal"]


print("orders_clean")
print(orders_clean[orders_num].describe())
print()
print(orders_clean[orders_cat].describe())
print()
print("catalog_clean")
print(catalog_clean[catalog_num].describe())
print()
print(catalog_clean[catalog_cat].describe())
print()
print("marketing_clean")
print(marketing_clean[marketing_num].describe())
print()
print(marketing_clean[marketing_cat].describe())

orders_clean
           cantidad  precio_unitario  monto_descuento   monto_total
count  24950.000000     24950.000000     24950.000000  2.500000e+04
mean       7.115110       259.374527         4.502605  2.079498e+03
std      296.869961       138.690738         5.224057  9.914760e+04
min       -2.000000        20.030000         0.000000 -4.926500e+02
25%        1.000000       138.470000         0.000000  1.806700e+02
50%        2.000000       258.795000         0.000000  3.418050e+02
75%        2.000000       380.377500        10.000000  5.185800e+02
max    20000.000000       499.960000        15.000000  8.840200e+06

          id_pedido id_usuario      pais dispositivo fuente_referencia  \
count         25000      25000     24700       24980             24970   
unique        25000       7642         6           2                 3   
top     order_23456  user_7769  Colombia     desktop            social   
freq              1         11      7481       12711              8402   

   

In [13]:
# Estandarización de país: se unifican mayúsculas para que coincida con marketing
print(orders_clean["pais"].value_counts())
orders_clean["pais"] = orders_clean["pais"].str.title()
print()
print(orders_clean["pais"].value_counts())
print()
print(marketing_clean["pais"].value_counts())

Colombia     7481
Mexico       7478
Argentina    7259
mexico        863
colombia      823
argentina     796
Name: pais, dtype: int64

Mexico       8341
Colombia     8304
Argentina    8055
Name: pais, dtype: int64

Argentina    540
Mexico       540
Colombia     540
Name: pais, dtype: int64


In [14]:
# Estandarización de categoría: se elimina la tilde para que coincida entre orders y catalog
print(orders_clean["categoria_producto"].value_counts())
print()
print(catalog_clean["categoria_producto"].value_counts())
print()
catalog_clean["categoria_producto"] = catalog_clean["categoria_producto"].replace("Electrónica", "Electronica")

print(catalog_clean["categoria_producto"].value_counts())

Hogar          8346
Moda           8295
Electronica    8279
Name: categoria_producto, dtype: int64

Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64

Electronica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64


In [15]:
# Recuperación de nulos en canal a partir de id_campaña y validación contra fuente_referencia
marketing_clean["canal"] = marketing_clean["canal"].fillna(
    marketing_clean["id_campaña"].str.rsplit("_", n=1).str[0]
)

print(marketing_clean["canal"].isna().sum())
print(marketing_clean["canal"].value_counts())
print()
print(orders_clean["fuente_referencia"].value_counts())

0
organic        540
social         540
paid_search    540
Name: canal, dtype: int64

social         8402
organic        8288
paid_search    8280
Name: fuente_referencia, dtype: int64


In [16]:
# Diagnóstico de valores inválidos y atípicos
cant_invalida = orders_clean["cantidad"] <= 0
monto_negativo = orders_clean["monto_total"] < 0

print("Pedidos con cantidad <= 0:", cant_invalida.sum())
print("Pedidos con monto_total negativo:", monto_negativo.sum())
print("Pedidos con ambos problemas:", (cant_invalida & monto_negativo).sum())
print()
print("Percentiles de cantidad")
print(orders_clean["cantidad"].quantile([0.50, 0.90, 0.95, 0.99, 0.999]))
print()
print("Cantidades más altas registradas")
print(orders_clean["cantidad"].value_counts().sort_index(ascending=False).head(10))

Pedidos con cantidad <= 0: 4
Pedidos con monto_total negativo: 4
Pedidos con ambos problemas: 4

Percentiles de cantidad
0.500    2.0
0.900    2.0
0.950    2.0
0.990    2.0
0.999    2.0
Name: cantidad, dtype: float64

Cantidades más altas registradas
 20000.0        4
 10000.0        6
 2.0        12591
 1.0        12345
-1.0            3
-2.0            1
Name: cantidad, dtype: int64


In [17]:
# Eliminación de valores inválidos y atípicos
umbral_cantidad = 10  # definir según el diagnóstico

filas_antes = len(orders_clean)

filtro_invalidos = (
    (orders_clean["cantidad"] <= 0)
    | (orders_clean["cantidad"] > umbral_cantidad)
    | (orders_clean["monto_total"] < 0)
)

orders_clean = orders_clean[~filtro_invalidos].reset_index(drop=True)

filas_despues = len(orders_clean)
eliminadas = filas_antes - filas_despues

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", eliminadas, f"({eliminadas / filas_antes:.2%})")
print()
print(orders_clean[orders_num].describe())

Filas antes: 25000
Filas después: 24986
Filas eliminadas: 14 (0.06%)

           cantidad  precio_unitario  monto_descuento   monto_total
count  24936.000000     24936.000000     24936.000000  24986.000000
mean       1.504933       259.362882         4.504331    385.972527
std        0.499986       138.681161         5.224549    255.686443
min        1.000000        20.030000         0.000000      5.240000
25%        1.000000       138.490000         0.000000    180.692500
50%        2.000000       258.775000         0.000000    341.710000
75%        2.000000       380.372500        10.000000    517.830000
max        2.000000       499.960000        15.000000    999.890000


In [18]:
# Verificar consistencia: monto_total = cantidad * precio_unitario - monto_descuento
monto_calculado = (
    orders_clean["cantidad"] * orders_clean["precio_unitario"]
    - orders_clean["monto_descuento"]
)
diferencia = (orders_clean["monto_total"] - monto_calculado).abs().round(2)
inconsistentes = diferencia > 0.01

print("Pedidos con monto inconsistente:", inconsistentes.sum())
print()
print(orders_clean.loc[inconsistentes, ["cantidad", "precio_unitario", "monto_descuento", "monto_total"]].head(10))

Pedidos con monto inconsistente: 0

Empty DataFrame
Columns: [cantidad, precio_unitario, monto_descuento, monto_total]
Index: []


In [19]:
# Nulos antes del tratamiento
print("Nulos antes:")
print(orders_clean.isna().sum())

# Categoría desde el catálogo
mapa_categoria = catalog_clean.set_index("nombre_producto")["categoria_producto"]
orders_clean["categoria_producto"] = orders_clean["categoria_producto"].fillna(
    orders_clean["nombre_producto"].map(mapa_categoria)
)

# Valores numéricos despejados de la fórmula del monto
orders_clean["cantidad"] = orders_clean["cantidad"].fillna(
    ((orders_clean["monto_total"] + orders_clean["monto_descuento"]) / orders_clean["precio_unitario"]).round()
)
orders_clean["precio_unitario"] = orders_clean["precio_unitario"].fillna(
    ((orders_clean["monto_total"] + orders_clean["monto_descuento"]) / orders_clean["cantidad"]).round(2)
)
orders_clean["monto_descuento"] = orders_clean["monto_descuento"].fillna(
    (orders_clean["cantidad"] * orders_clean["precio_unitario"] - orders_clean["monto_total"]).round(2)
)

print()
print("Nulos después de recuperar:")
print(orders_clean.isna().sum())

Nulos antes:
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

Nulos después de recuperar:
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  296
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64


In [20]:
# Tratamiento final de nulos
filas_antes = len(orders_clean)

# Sin producto o sin valores numéricos no se puede calcular costo ni revenue: se eliminan
orders_clean = orders_clean.dropna(
    subset=["nombre_producto", "categoria_producto", "cantidad", "precio_unitario", "monto_descuento"]
).reset_index(drop=True)

# Variables descriptivas faltantes: se conservan como "Desconocido"
for col in ["pais", "dispositivo", "fuente_referencia"]:
    orders_clean[col] = orders_clean[col].fillna("Desconocido")

filas_despues = len(orders_clean)
eliminadas = filas_antes - filas_despues

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", eliminadas, f"({eliminadas / filas_antes:.2%})")
print()
print("Nulos restantes:", orders_clean.isna().sum().sum())

# Cantidad como entero
orders_clean["cantidad"] = orders_clean["cantidad"].astype(int)

Filas antes: 24986
Filas después: 24906
Filas eliminadas: 80 (0.32%)

Nulos restantes: 0


In [21]:
# Validación final de los tres datasets limpios
for nombre, df, num, cat in [
    ("orders_clean", orders_clean, orders_num, orders_cat),
    ("catalog_clean", catalog_clean, catalog_num, catalog_cat),
    ("marketing_clean", marketing_clean, marketing_num, marketing_cat),
]:
    print("=" * 60)
    print(nombre)
    print("=" * 60)
    df.info()
    print()
    print(df[num].describe())
    print()
    print(df[cat].describe())
    print()

orders_clean
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24906 entries, 0 to 24905
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24906 non-null  object        
 1   id_usuario          24906 non-null  object        
 2   fecha_hora_pedido   24906 non-null  datetime64[ns]
 3   pais                24906 non-null  object        
 4   dispositivo         24906 non-null  object        
 5   fuente_referencia   24906 non-null  object        
 6   nombre_producto     24906 non-null  object        
 7   categoria_producto  24906 non-null  object        
 8   cantidad            24906 non-null  int64         
 9   precio_unitario     24906 non-null  float64       
 10  monto_descuento     24906 non-null  float64       
 11  monto_total         24906 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(7)
memory usage: 2.3+ MB

           c

**Exportación:** los datasets limpios se exportan para construir el dashboard en Power BI (sección 6).

In [22]:
# Exportar los datasets limpios
orders_clean.to_csv('orders_clean.csv', index=False)
catalog_clean.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

### Conclusión: ¿podemos confiar en los datos?

**Sí.** Después de la limpieza, los tres datasets son confiables y están listos para el análisis. Los problemas encontrados se corrigieron o se eliminaron con una pérdida mínima de información.

**orders: de 25 100 a 24 906 registros (-0,77 %)**
- Se eliminaron **100 pedidos duplicados** (filas idénticas con el mismo `id_pedido`).
- Se eliminaron **14 pedidos con valores inválidos**: 4 con cantidad y monto negativos, y 10 con cantidades atípicas (10 000 y 20 000 unidades). El 99,9 % de los pedidos tiene entre 1 y 2 unidades, por lo que se definió un umbral máximo de 10.
- Se eliminaron **80 pedidos sin producto o sin valores numéricos**, ya que sin ellos no es posible calcular revenue ni costo. En esas filas faltaban `cantidad`, `precio_unitario` y `monto_descuento` al mismo tiempo, por lo que no se pudieron deducir de la fórmula del monto.
- Se recuperaron **50 categorías** a partir del catálogo usando el nombre del producto.
- Los nulos de `pais` (296) y `dispositivo` (20) se reemplazaron por **"Desconocido"**. Estos pedidos tienen información válida de producto y monto, y eliminarlos subestimaría el revenue y el profit.
- `pais` se estandarizó de 6 a 3 valores (diferencias de mayúsculas) y coincide con marketing.
- `fecha_hora_pedido` se convirtió a `datetime` y `cantidad` a entero.
- No se encontraron inconsistencias entre `monto_total` y `cantidad × precio_unitario − monto_descuento`.

**catalog: 7 productos**
- Sin nulos ni duplicados. Se unificó el formato de la categoría ("Electrónica" → "Electronica") para que coincida con orders.

**marketing: 1 620 registros**
- Sin duplicados (combinación fecha + campaña): 9 campañas × 180 días.
- Se recuperaron **101 valores nulos de `canal`** a partir de `id_campaña`, sin eliminar registros de gasto.
- `fecha` se convirtió a `datetime`. Los canales coinciden con `fuente_referencia` en orders.

**Consideraciones para los siguientes pasos**
- Los pedidos cubren del 1 de enero al 30 de junio de 2025, mientras que el gasto de marketing llega hasta el 29 de junio. El último día tendrá ventas sin inversión registrada.
- Los pedidos con país o dispositivo "Desconocido" deben tenerse en cuenta en los análisis que se segmenten por esas variables.

---
## 2. Rentabilidad del negocio

Se calculan los indicadores clave para evaluar ingresos, costos y rentabilidad: revenue, costo de los productos, inversión en marketing, profit y margen; luego, el comportamiento de ventas (ticket promedio, productos por orden, producto más vendido) y la rentabilidad por categoría, canal y país.

In [23]:
# Unión de pedidos y catálogo para calcular el costo de cada orden
unido = pd.merge(
    orders_clean,
    catalog_clean,
    on=["nombre_producto", "categoria_producto"],
    how="left"
)
unido["costo_orden"] = unido["cantidad"] * unido["costo_unitario"]

print("Filas en orders_clean:", len(orders_clean))
print("Filas en unido:", len(unido))
print("Pedidos sin costo asignado:", unido["costo_unitario"].isna().sum())
display(unido.head(5))

Filas en orders_clean: 24906
Filas en unido: 24906
Pedidos sin costo asignado: 0


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor,costo_orden
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2,332.69,0.0,665.37,189.31,Mcmillan-Rhodes,378.62
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1,176.86,5.0,171.86,25.21,Bowers LLC,25.21
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2,102.99,10.0,195.99,176.64,Long-Reid,353.28
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1,257.87,15.0,242.87,25.21,Bowers LLC,25.21
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1,336.28,0.0,336.28,176.64,Long-Reid,176.64


In [24]:
# KPIs de rentabilidad del negocio
revenue = orders_clean["monto_total"].sum()
costo_total = unido["costo_orden"].sum()
inversion_mark = marketing_clean["gasto"].sum()
profit = revenue - costo_total - inversion_mark
margen_profit = profit / revenue * 100

print(f"Revenue:                  ${revenue:,.2f}")
print(f"Costo total:              ${costo_total:,.2f}")
print(f"Inversión en marketing:   ${inversion_mark:,.2f}")
print(f"Profit:                   ${profit:,.2f}")
print(f"Margen de profit:         {margen_profit:.2f}%")

Revenue:                  $9,610,018.94
Costo total:              $3,828,869.01
Inversión en marketing:   $2,871,843.53
Profit:                   $2,909,306.40
Margen de profit:         30.27%


In [25]:
# Comportamiento de ventas: ticket y cantidad promedio por orden
ticket_prom = revenue / orders_clean["id_pedido"].count()
cantidad_prom = orders_clean["cantidad"].mean()

print(f"Ticket promedio:                  ${ticket_prom:,.2f}")
print(f"Cantidad promedio:                  {cantidad_prom:,.2f}")

Ticket promedio:                  $385.85
Cantidad promedio:                  1.50


In [26]:
# Comportamiento de ventas: producto más vendido y gasto de marketing por canal
print("Unidades vendidas por producto")
print(orders_clean.groupby("nombre_producto")["cantidad"].sum().sort_values(ascending=False))
print()
print("Gasto en marketing por canal")
print(marketing_clean.groupby("canal")["gasto"].sum().sort_values(ascending=False).round(2))

Unidades vendidas por producto
nombre_producto
Vacuum-Pro-Black        6284
Blender-XL-Red          6279
Jacket-Winter-M         6256
Sneakers-Urban-42       6172
Laptop-Gaming-16GB      4198
Tablet-Standard-64GB    4153
Phone-Pro-128GB         4140
Name: cantidad, dtype: int64

Gasto en marketing por canal
canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64


In [27]:
# Segmentos rentables por categoría (ganancia bruta: el marketing no está asignado a productos)
seg_categoria = unido.groupby("categoria_producto").agg(
    revenue=("monto_total", "sum"),
    costo=("costo_orden", "sum")
)
seg_categoria["ganancia_bruta"] = seg_categoria["revenue"] - seg_categoria["costo"]
seg_categoria["margen_%"] = seg_categoria["ganancia_bruta"] / seg_categoria["revenue"] * 100
seg_categoria = seg_categoria.sort_values("ganancia_bruta", ascending=False).round(2)

print("Rentabilidad por categoría")
display(seg_categoria)

Rentabilidad por categoría


,revenue,costo,ganancia_bruta,margen_%
categoria_producto,,,,
Hogar,3232626.44,1213436.96,2019189.48,62.46
Moda,3172662.10,1290543.48,1882118.62,59.32
Electronica,3204730.40,1324888.57,1879841.83,58.66


In [28]:
# Segmentos rentables por canal (incluye el gasto de marketing de cada canal)
ventas_canal = unido.groupby("fuente_referencia").agg(
    revenue=("monto_total", "sum"),
    costo=("costo_orden", "sum")
)
ventas_canal.index.name = "canal"

gasto_canal = marketing_clean.groupby("canal").agg(gasto_marketing=("gasto", "sum"))

seg_canal = ventas_canal.join(gasto_canal, how="left").fillna({"gasto_marketing": 0})
seg_canal["profit"] = seg_canal["revenue"] - seg_canal["costo"] - seg_canal["gasto_marketing"]
seg_canal["margen_%"] = seg_canal["profit"] / seg_canal["revenue"] * 100
seg_canal = seg_canal.sort_values("profit", ascending=False).round(2)

print("Rentabilidad por canal")
display(seg_canal)

Rentabilidad por canal


,revenue,costo,gasto_marketing,profit,margen_%
canal,,,,,
paid_search,3209280.07,1240959.06,922374.20,1045946.81,32.59
social,3222510.32,1312735.79,976818.37,932956.16,28.95
organic,3178228.55,1275174.16,972650.96,930403.43,29.27


In [29]:
# Segmentos rentables por país (incluye el gasto de marketing de cada país)
ventas_pais = unido.groupby("pais").agg(
    revenue=("monto_total", "sum"),
    costo=("costo_orden", "sum")
)

gasto_pais = marketing_clean.groupby("pais").agg(gasto_marketing=("gasto", "sum"))

seg_pais = ventas_pais.join(gasto_pais, how="left").fillna({"gasto_marketing": 0})
seg_pais["profit"] = seg_pais["revenue"] - seg_pais["costo"] - seg_pais["gasto_marketing"]
seg_pais["margen_%"] = seg_pais["profit"] / seg_pais["revenue"] * 100
seg_pais = seg_pais.sort_values("profit", ascending=False).round(2)

print("Rentabilidad por país")
print("Nota: 'Desconocido' no tiene gasto de marketing asociado, por lo que su margen aparece inflado.")
display(seg_pais)

Rentabilidad por país
Nota: 'Desconocido' no tiene gasto de marketing asociado, por lo que su margen aparece inflado.


,revenue,costo,gasto_marketing,profit,margen_%
pais,,,,,
Colombia,3170333.82,1270630.14,935653.42,964050.26,30.41
Mexico,3207047.74,1278497.91,988495.51,940054.32,29.31
Argentina,3120649.33,1236031.22,947694.60,936923.51,30.02
Desconocido,111988.05,43709.74,0.00,68278.31,60.97


### Conclusión: ¿el negocio genera ganancias?

**Sí. RappiPlus es rentable**, con un profit de **USD 2 909 306,40** y un margen del **30,27 %** sobre un revenue de **USD 9 610 018,94**.

**KPIs principales**

| Indicador | Valor |
|---|---|
| Revenue | USD 9 610 018,94 |
| Costo de productos | USD 3 828 869,01 |
| Inversión en marketing | USD 2 871 843,53 |
| Profit | USD 2 909 306,40 |
| Margen de profit | 30,27 % |

- El margen bruto (antes de marketing) es del **60,2 %**: los productos se venden muy por encima de su costo.
- El marketing consume el **29,9 % del revenue**, casi lo mismo que el profit final. Es el principal factor que reduce la rentabilidad.

**Comportamiento de ventas**
- El **ticket promedio** es de **USD 385,85** por orden, con **1,5 productos** en promedio.
- **Vacuum-Pro-Black** es el producto más vendido (6 284 unidades), aunque prácticamente empata con Blender-XL-Red, Jacket-Winter-M y Sneakers-Urban-42. Los productos de electrónica forman un segundo nivel de demanda, con unas 4 150 unidades cada uno.
- El gasto en marketing está repartido de forma casi uniforme entre canales (entre USD 922 000 y USD 977 000). Llama la atención que **organic** tenga gasto asignado; conviene validar con el negocio qué incluye (por ejemplo, SEO o contenido).

**Segmentos rentables**
- **Categoría:** las tres generan un revenue similar (unos USD 3,2 M), pero **Hogar** es la más rentable, con la mayor ganancia bruta (USD 2,02 M) y el mejor margen (62,5 %). Electrónica tiene el costo más alto y el menor margen (58,7 %).
- **Canal:** **paid_search** es el canal más eficiente: genera el mayor profit (USD 1,05 M) y el mejor margen (32,6 %) con la menor inversión. Por cada USD 1 invertido deja USD 1,13 de profit, frente a USD 0,96 de social y organic. **Social** recibe la mayor inversión, pero tiene el peor margen (29,0 %).
- **País:** **Colombia** lidera en profit (USD 964 000) y margen (30,4 %). **México** factura más y recibe más inversión, pero obtiene el menor margen (29,3 %). Los pedidos con país "Desconocido" se excluyen de la comparación porque no tienen gasto de marketing asociado.

**Recomendaciones**
- **Reasignar presupuesto de marketing de social hacia paid_search**, que ofrece el mejor retorno por cada dólar invertido.
- **Revisar la eficiencia del gasto en México**, donde una mayor inversión no se traduce en un mejor margen.
- **Potenciar la categoría Hogar**, que combina buen volumen de ventas con el mayor margen.
- Ningún segmento genera pérdidas: las mejoras están en optimizar la eficiencia del marketing, no en corregir áreas deficitarias.

---
## 3. Funnel de conversión

Con consultas SQL sobre la tabla `events` se cuenta cuántos usuarios únicos llegan a cada etapa, se calcula la conversión entre pasos y se identifica dónde se pierde la mayor cantidad de usuarios.

In [30]:
import os
import pandas as pd
from sqlalchemy import create_engine

# Conexión a la base de datos PostgreSQL del caso de estudio.
# Las credenciales se leen de variables de entorno y no se publican en el repositorio.
db_config = {
    'user': os.environ['DB_USER'],
    'pwd': os.environ['DB_PASSWORD'],
    'host': os.environ['DB_HOST'],
    'port': int(os.environ.get('DB_PORT', 5432)),
    'db': os.environ['DB_NAME']
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [31]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [32]:
# Funnel: usuarios únicos por etapa, ordenados según el flujo.
query_funnel = '''

SELECT
    nombre_evento,
    COUNT(DISTINCT(id_usuario)) AS Usuarios
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
    END;
'''
funnel = pd.read_sql(query_funnel, con=engine)
funnel

,nombre_evento,usuarios
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [33]:
# Conversión entre etapas del funnel
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS etapa,
        COUNT(DISTINCT id_usuario) AS usuarios
    FROM events
    GROUP BY nombre_evento
)
SELECT
    etapa,
    nombre_evento,
    usuarios,
    LAG(usuarios) OVER (ORDER BY etapa) AS usuarios_etapa_anterior,
    ROUND(usuarios * 100.0 / LAG(usuarios) OVER (ORDER BY etapa), 1) AS conversion_paso_pct,
    ROUND(100 - usuarios * 100.0 / LAG(usuarios) OVER (ORDER BY etapa), 1) AS perdida_paso_pct,
    ROUND(usuarios * 100.0 / FIRST_VALUE(usuarios) OVER (ORDER BY etapa), 1) AS conversion_acumulada_pct
FROM funnel
ORDER BY etapa;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,etapa,nombre_evento,usuarios,usuarios_etapa_anterior,conversion_paso_pct,perdida_paso_pct,conversion_acumulada_pct
0,1,first_visit,7796,NaN,NaN,NaN,100.0
1,2,select_item,7582,7796.0,97.3,2.7,97.3
2,3,add_to_cart,7634,7582.0,100.7,-0.7,97.9
3,4,begin_checkout,7208,7634.0,94.4,5.6,92.5
4,5,add_payment_info,6250,7208.0,86.7,13.3,80.2
5,6,purchase,6240,6250.0,99.8,0.2,80.0


### Conclusión: ¿dónde se pierden los usuarios?

**El funnel tiene una conversión final del 80,0 %**: de 7 796 usuarios que hicieron su primera visita, 6 240 completaron una compra.

**Funnel de conversión**

| Etapa | Evento | Usuarios | Conversión entre pasos | Pérdida | Conversión acumulada |
|---|---|---|---|---|---|
| 1 | first_visit | 7 796 | – | – | 100,0 % |
| 2 | select_item | 7 582 | 97,3 % | 2,7 % | 97,3 % |
| 3 | add_to_cart | 7 634 | 100,7 % | -0,7 % | 97,9 % |
| 4 | begin_checkout | 7 208 | 94,4 % | 5,6 % | 92,5 % |
| 5 | add_payment_info | 6 250 | 86,7 % | **13,3 %** | 80,2 % |
| 6 | purchase | 6 240 | 99,8 % | 0,2 % | 80,0 % |

**Principal cuello de botella**
- La mayor pérdida ocurre **entre el inicio del checkout y el ingreso de los datos de pago**: **958 usuarios (13,3 %)** abandonan en ese paso, más del doble que en cualquier otra etapa.
- En cambio, una vez que el usuario ingresa sus datos de pago, prácticamente todos compran (99,8 %). El problema no está en la compra final, sino en el **momento de solicitar la información de pago**.
- La segunda pérdida más relevante está entre agregar al carrito e iniciar el checkout (5,6 %).

**Etapas iniciales**
- Las primeras etapas tienen pérdidas bajas: el 97,3 % de los visitantes selecciona un producto, lo que indica que el tráfico que llega a la plataforma es relevante.

**Nota sobre los datos**
- `add_to_cart` registra más usuarios que `select_item` (conversión del 100,7 %). Esto se debe a que el análisis es un **funnel abierto**, que cuenta a todos los usuarios que realizaron cada evento sin exigir que hayan pasado por las etapas anteriores. Algunos usuarios pueden agregar productos al carrito sin entrar al detalle del producto, por ejemplo desde un listado o una recomendación.

**Recomendaciones**
- **Priorizar la optimización del paso de pago**: revisar la longitud del formulario, los métodos de pago disponibles, la aparición de costos inesperados (envío, impuestos) y la confianza que transmite esa pantalla.
- **Implementar recordatorios para los usuarios que abandonan el checkout**, ya que llegaron muy cerca de la compra.
- **Validar cualquier cambio en el checkout con un experimento A/B**, como se analiza en la sección 5, para medir su impacto real antes de implementarlo.

---
## 4. Retención por cohortes

Con las tablas `users` y `user_activity` se agrupa a los usuarios por **mes de registro** y se calcula qué porcentaje de cada cohorte sigue activo en las semanas 1 a 4 después de registrarse. La fecha de registro se convierte con `CAST(fecha_registro AS DATE)`.

In [34]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [35]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [36]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        fecha_registro,
        TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'YYYY-MM') AS cohorte
    FROM users
),
retencion AS (
    SELECT
        ch.cohorte,
        COUNT(DISTINCT ch.id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 7 AND ua.activo = 1 THEN ch.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 14 AND ua.activo = 1 THEN ch.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 21 AND ua.activo = 1 THEN ch.id_usuario END) AS retenido_w3,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 28 AND ua.activo = 1 THEN ch.id_usuario END) AS retenido_w4
    FROM cohortes AS ch
    LEFT JOIN user_activity AS ua ON ch.id_usuario = ua.id_usuario
    GROUP BY ch.cohorte
)
SELECT
    cohorte,
    clientes_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    retenido_w4,
    ROUND(retenido_w1 * 100.0 / clientes_iniciales, 1) AS semana_1,
    ROUND(retenido_w2 * 100.0 / clientes_iniciales, 1) AS semana_2,
    ROUND(retenido_w3 * 100.0 / clientes_iniciales, 1) AS semana_3,
    ROUND(retenido_w4 * 100.0 / clientes_iniciales, 1) AS semana_4
FROM retencion
ORDER BY cohorte;
'''
# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,retenido_w4,semana_1,semana_2,semana_3,semana_4
0,2025-01,1627,697,668,656,671,42.8,41.1,40.3,41.2
1,2025-02,1444,611,609,635,575,42.3,42.2,44.0,39.8
2,2025-03,1636,677,705,690,673,41.4,43.1,42.2,41.1
3,2025-04,1606,680,697,663,652,42.3,43.4,41.3,40.6
4,2025-05,1687,695,676,706,679,41.2,40.1,41.8,40.2


### Conclusión: ¿los usuarios regresan?

**Sí, pero solo una parte.** En promedio, cerca del **42 % de los usuarios está activo cada semana** después de registrarse, y ese nivel se mantiene estable en todas las cohortes.

**Retención semanal por cohorte (%)**

| Cohorte | Clientes iniciales | Semana 1 | Semana 2 | Semana 3 | Semana 4 |
|---|---|---|---|---|---|
| 2025-01 | 1 627 | 42,8 | 41,1 | 40,3 | 41,2 |
| 2025-02 | 1 444 | 42,3 | 42,2 | 44,0 | 39,8 |
| 2025-03 | 1 636 | 41,4 | 43,1 | 42,2 | 41,1 |
| 2025-04 | 1 606 | 42,3 | 43,4 | 41,3 | 40,6 |
| 2025-05 | 1 687 | 41,2 | 40,1 | 41,8 | 40,2 |
| **Promedio** | | **42,0** | **42,0** | **41,9** | **40,6** |

**Hallazgos principales**
- **La retención es estable:** todas las cohortes se mantienen entre el **39,8 % y el 44,0 %** en las cuatro semanas. No hay una caída fuerte después del registro, como suele ocurrir en otras plataformas.
- **Hay un desgaste leve con el tiempo:** el promedio se mantiene en torno al 42 % durante las tres primeras semanas y baja al **40,6 %** en la semana 4.
- **Ninguna cohorte se destaca de forma consistente.** Febrero tiene la mejor semana individual (44,0 % en la semana 3), pero también la más baja (39,8 % en la semana 4). Enero es la única cohorte con una caída continua en las tres primeras semanas.
- **La oportunidad está en los inactivos:** casi **6 de cada 10 usuarios no están activos** en una semana dada, lo que representa un amplio margen para aumentar la frecuencia de uso.

**Nota sobre la metodología**
- La columna `activo` indica si el usuario estuvo activo **en cada semana por separado**, no si se mantuvo activo desde el registro. Por eso la retención puede subir de una semana a otra: un usuario puede estar inactivo en la semana 1 y volver en la semana 2.
- Se verificó que `user_activity` registra un corte por semana (días 7, 14, 21 y 28 después del registro). La semana 4 se incluyó como análisis complementario para confirmar la tendencia.

**Recomendaciones**
- **Implementar acciones de reactivación** (notificaciones, recordatorios o beneficios exclusivos de RappiPlus) dirigidas a los usuarios inactivos en sus primeras semanas.
- **Analizar qué diferencia a los usuarios activos de los inactivos** (país, dispositivo, canal de adquisición o tipo de plan) para enfocar esas acciones en los segmentos con menor retención.
- **Monitorear la retención más allá de la semana 4** para confirmar si el desgaste leve se acelera en los meses siguientes.

---
## 5. Experimento A/B del checkout

Se evalúa si una nueva interfaz del checkout cambia la **tasa de conversión de compra**. La métrica `convirtio` vale 1 si el usuario completó la compra y 0 si no.

**Hipótesis estadística**

- **H₀ (Hipótesis nula):** la tasa de conversión del grupo tratamiento (nueva UI del checkout) es igual a la del grupo control. El cambio no tiene efecto sobre la conversión.
- **H₁ (Hipótesis alternativa):** la tasa de conversión del grupo tratamiento es diferente a la del grupo control. El cambio sí tiene efecto sobre la conversión.

**Test estadístico:** prueba z para la diferencia de dos proporciones (bilateral), adecuada porque la métrica `convirtio` es binaria y se comparan dos grupos independientes.

**Nivel de significancia alpha:** 0,05

In [37]:
# Validación del experimento: estructura, duplicados y tamaño de grupos
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
display(df.head(5))
df.info()
print()
print("id duplicados")
print(df.duplicated("id_usuario").sum())
print()
print(df.value_counts("variante"))

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB

id duplicados
0

variante
tratamiento    5035
control        4965
dtype: int64


In [38]:
# Tasa de conversión por variante
conversion_tra = df[df["variante"] == "tratamiento"]["convirtio"].mean()
conversion_con = df[df["variante"] == "control"]["convirtio"].mean()

print(f"Conversión tratamiento: {conversion_tra:.2%}")
print(f"Conversión control:     {conversion_con:.2%}")
print(f"Diferencia:             {(conversion_tra - conversion_con) * 100:.2f} puntos porcentuales")

Conversión tratamiento: 16.29%
Conversión control:     15.69%
Diferencia:             0.60 puntos porcentuales


In [39]:
# Prueba z de dos proporciones: tratamiento vs. control
exitos_por_grupo = df.groupby("variante")["convirtio"].sum()
total_por_grupo = df.groupby("variante")["convirtio"].count()

exitos = [exitos_por_grupo["tratamiento"], exitos_por_grupo["control"]]
observaciones = [total_por_grupo["tratamiento"], total_por_grupo["control"]]

print(f"Conversiones (tratamiento, control): {exitos}")
print(f"Usuarios (tratamiento, control):     {observaciones}")
print()

z_stat, p_value = proportions_ztest(exitos, observaciones)

print(f"Estadístico z: {z_stat:.3f}")
print(f"Valor p:       {p_value:.4f}")
print()

alpha = 0.05

if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.")

Conversiones (tratamiento, control): [820, 779]
Usuarios (tratamiento, control):     [5035, 4965]

Estadístico z: 0.813
Valor p:       0.4161

No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.


### Conclusión: ¿el cambio genera impacto?

**No hay evidencia suficiente de que la nueva UI del checkout mejore la conversión de compra.**

**Resultados del experimento**

| Variante | Usuarios | Conversiones | Tasa de conversión |
|---|---|---|---|
| Control (UI actual) | 4 965 | 779 | 15,69 % |
| Tratamiento (nueva UI) | 5 035 | 820 | 16,29 % |
| **Diferencia** | | | **+0,60 p.p. (+3,8 % relativo)** |

- **Estadístico z:** 0,813
- **Valor p:** 0,4161
- **Nivel de significancia (α):** 0,05

**Interpretación**
- Como el **valor p (0,4161) es mayor que α (0,05)**, no se rechaza la hipótesis nula. La diferencia de 0,6 puntos porcentuales a favor del tratamiento **no es estadísticamente significativa**: con estos datos, puede explicarse por la variación aleatoria entre grupos.
- El experimento está bien construido: los grupos están balanceados (50,4 % frente a 49,6 %), no hay usuarios duplicados ni presentes en ambas variantes y no hay valores nulos. Por eso, el resultado se puede atribuir a la ausencia de un efecto detectable y no a problemas en los datos.

**Relación con el funnel (sección 3)**
- El funnel mostró que la mayor pérdida de usuarios ocurre en el checkout, entre el inicio del pago y el ingreso de los datos de pago (13,3 %). Este cambio en la UI **no logró resolver ese cuello de botella** de forma medible.

**Recomendaciones**
- **No implementar la nueva UI esperando un aumento en la conversión**, ya que su efecto no está demostrado. Si se decide adoptarla por otras razones (diseño, mantenimiento o experiencia de usuario), debe hacerse sabiendo que no hay evidencia de impacto en las ventas.
- **Si el equipo considera que el efecto es real pero pequeño**, repetir el experimento con una muestra mayor, ya que detectar una mejora de 0,6 puntos requiere muchos más usuarios por grupo.
- **Diseñar el próximo experimento enfocado en el paso de pago**, donde está la mayor pérdida del funnel: por ejemplo, simplificar el formulario, ofrecer más métodos de pago o mostrar los costos totales desde el inicio.

---
## 6. Dashboard en Power BI

El dashboard se construyó con los tres CSV limpios de la sección 1. El archivo [`dashboard_rappiplus.pbix`](dashboard_rappiplus.pbix) está en este repositorio.

### Preparación de los datos en Power BI

- Se cargaron los tres CSV limpios de la sección 1: `orders_clean.csv` (24 906 pedidos), `catalog_clean.csv` (7 productos) y `marketing_clean.csv` (1 620 registros de gasto).
- **Tipos de datos en Power Query:** las columnas numéricas con punto decimal (`precio_unitario`, `monto_descuento`, `monto_total`, `costo_unitario` y `gasto`) se convirtieron a número decimal con la configuración regional Inglés (Estados Unidos), ya que con la configuración por defecto Power BI eliminaba el punto (665.37 se leía como 66537). `cantidad` quedó como número entero y las fechas como tipo Fecha. Ninguna columna presenta errores ni vacíos, y `nombre_producto` es único en el catálogo (7 valores distintos en 7 filas).
- **Tabla calendario (`Dim_Fecha`):** creada con DAX (`CALENDAR` + `ADDCOLUMNS`) del 1 de enero al 30 de junio de 2025, con Año, Mes, Mes Numero y Año-Mes. Se marcó como tabla de fechas y el mes se ordenó por su número. Permite los cálculos YTD; las comparaciones YoY no aplican porque los datos cubren solo el primer semestre de 2025.
- **Modelo estrella**, con relaciones 1:*, activas y de dirección única:
  - `catalog_clean[nombre_producto]` → `orders_clean[nombre_producto]`, para obtener el costo unitario de cada pedido.
  - `Dim_Fecha[Date]` → `orders_clean[fecha_hora_pedido]`, para el análisis temporal de ventas.
  - `Dim_Fecha[Date]` → `marketing_clean[fecha]`, para que el gasto en marketing también se filtre por fecha y el profit mensual se calcule correctamente.
  - Las columnas de fecha y producto de las tablas de hechos se ocultaron para filtrar siempre desde las dimensiones.
- **Medidas DAX principales:** Revenue (`SUM` de `monto_total`), Costo Total (`SUMX` de cantidad × costo unitario con `RELATED`), Gasto Marketing, Ganancia Bruta (revenue − costo), Profit (revenue − costo − marketing), Margen, Ticket Promedio, Cantidad Promedio, Revenue y Profit YTD (`TOTALYTD`), y Pedidos con Pérdida (`FILTER` sobre los pedidos con ganancia bruta negativa). Los totales del modelo coinciden con los calculados en Python en la sección 2.



### Conclusión: dashboard de resultados

El dashboard se construyó en Power BI con tres páginas: **Overview** (visión ejecutiva), **Detalle** (diagnóstico por producto y pedido) y **Pedidos por producto** (destino del drill-through, oculta y accesible con clic derecho sobre un producto en la página Detalle).

**Overview: el negocio es rentable, pero el marketing consume casi tanto como la ganancia**
- RappiPlus generó **USD 9,61 M** de revenue en el primer semestre de 2025, con un **profit de USD 2,91 M** (margen del **30,3 %**).
- El **gasto en marketing (USD 2,87 M) equivale al 29,9 % del revenue**, prácticamente lo mismo que el profit final.
- Las ventas son estables: el revenue mensual se mueve entre USD 1,44 M y 1,66 M. La caída de febrero se explica principalmente por tener menos días; su venta diaria es solo un 3,5 % menor al promedio.
- Las tres categorías aportan un revenue similar (unos USD 3,2 M cada una). Hogar tiene el mejor margen bruto (62,5 %).
- El ticket promedio es de USD 385,85, con 1,5 productos por pedido.

**Detalle: casi 1 de cada 5 pedidos se vende con pérdida**
- **4 528 pedidos (18,2 %)** se vendieron por debajo de su costo.
- Las pérdidas se concentran en tres productos, que suman el **98 %** de estos casos: **Jacket-Winter-M** (1 525 pedidos, 36,6 % de sus ventas), **Laptop-Gaming-16GB** (1 498 pedidos, 54,1 %) y **Blender-XL-Red** (1 414 pedidos, 33,9 %). Son los únicos productos con costo unitario superior a 175.
- **Laptop-Gaming-16GB pierde dinero en el semestre**, incluso antes de restar marketing: revenue de USD 1 084 673 frente a un costo de USD 1 178 295 (**profit bruto de −USD 93 622**).
- La causa es que el precio de venta no refleja el costo del producto: todos los productos se venden en un rango similar de precios (entre unos 20 y 500), mientras que su costo es fijo. Los productos caros quedan expuestos a ventas por debajo del costo.

**Recomendaciones**
- **Revisar la política de precios y descuentos**, estableciendo un precio mínimo por producto que cubra su costo, empezando por Laptop-Gaming-16GB, Jacket-Winter-M y Blender-XL-Red.
- **Validar con el equipo comercial** si las ventas por debajo del costo responden a promociones intencionales o a errores de precio.
- **Optimizar la inversión en marketing**, que consume casi el 30 % del revenue: reasignar presupuesto hacia paid_search, el canal más eficiente según la sección 2.